# Assignment 05 – Evaluation: Beyond Accuracy
**Domain:** Cybersecurity – Clinic Security Monitoring  
**Positive class:** Phishing (1); legitimate (0).

This notebook calculates a confusion matrix, precision, recall and F1; compares false-positive and false-negative costs; runs 5-fold cross-validation; uses preprocessing pipelines; and compares at least three models.

## 1. Load dataset and prepare target/features
The official UCI labels are recoded so phishing is the positive class. Only numeric URL-structure features are used for the baseline comparison.

In [ ]:
# Load the official UCI PhiUSIIL phishing URL dataset.
# UCI label convention: 1 = legitimate, 0 = phishing.
!pip -q install ucimlrepo

import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=967)
df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)
df.columns = [str(c).strip() for c in df.columns]
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Label column candidates:", [c for c in df.columns if c.lower() == "label"])
display(df.head())


label_col = next(c for c in df.columns if c.lower() == "label")
# UCI convention: 0 = phishing, 1 = legitimate. Recode phishing as positive class (1).
y_all = (pd.to_numeric(df[label_col], errors="coerce") == 0).astype(int)

# Use numeric URL-structure features only; exclude identifiers and target-derived similarity/probability fields.
preferred = [
    "URLLength", "DomainLength", "IsDomainIP", "NoOfSubDomain",
    "NoOfObfuscatedChar", "NoOfLettersInURL", "LetterRatioInURL",
    "NoOfDegitsInURL", "DegitRatioInURL", "NoOfEqualsInURL",
    "NoOfQMarkInURL", "NoOfAmpersandInURL", "NoOfOtherSpecialCharsInURL",
    "IsHTTPS", "NoOfURLRedirect", "NoOfSelfRedirect"
]
available = {c.lower(): c for c in df.columns}
feature_cols = [available[c.lower()] for c in preferred if c.lower() in available]
if len(feature_cols) < 3:
    excluded = {"label", "filename", "url", "domain", "tld", "title",
                "urlsimilarityindex", "urltitlematchscore", "urlcharprob",
                "tldlegitimateprob"}
    feature_cols = [
        c for c in df.select_dtypes(include=np.number).columns
        if c.lower() not in excluded and c.lower() != label_col.lower()
    ][:10]

X_all = df[feature_cols].apply(pd.to_numeric, errors="coerce")
print("Target: phishing (1) vs legitimate (0)")
print("Features:", feature_cols)
print("Class balance:")
display(y_all.value_counts(normalize=True).rename("proportion").to_frame())

## 2. Train/test split and three leakage-safe pipelines
Imputation and scaling are fitted inside each pipeline, so the test fold does not influence preprocessing.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

sample_size = min(15000, len(X_all))
sample_parts = []
for cls, group in y_all.groupby(y_all):
    n_cls = max(1, round(sample_size * len(group) / len(y_all)))
    sample_parts.append(group.sample(n=min(n_cls, len(group)), random_state=42).index)
sample_index = pd.Index(np.concatenate(sample_parts))
X, y = X_all.loc[sample_index], y_all.loc[sample_index]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(max_depth=8, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1))
    ])
}
print("Train rows:", len(X_train), "| Test rows:", len(X_test))

## 3. Fit models and compare the same test metrics
For each model, calculate accuracy, precision, recall and F1 on the same held-out test set.

In [ ]:
results = []
fitted_models = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    fitted_models[name] = model
    predictions[name] = pred
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0)
    })

evaluation_table = pd.DataFrame(results).set_index("Model").sort_values("F1", ascending=False)
display(evaluation_table.round(4))

## 4. Confusion matrix for the best F1 model
Read the four cells explicitly: true negatives, false positives, false negatives and true positives. Rows are actual classes; columns are predicted classes.

In [ ]:
best_name = evaluation_table.index[0]
best_model = fitted_models[best_name]
best_predictions = predictions[best_name]
cm = confusion_matrix(y_test, best_predictions, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

print("Selected model:", best_name)
print("Confusion matrix (rows=actual [legitimate, phishing], columns=predicted [legitimate, phishing]):")
print(cm)
print("True negatives (legitimate correctly identified):", tn)
print("False positives (legitimate incorrectly flagged as phishing):", fp)
print("False negatives (phishing incorrectly classified as legitimate):", fn)
print("True positives (phishing correctly identified):", tp)

import matplotlib.pyplot as plt
plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title(f"Confusion Matrix – {best_name}")
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.xticks([0, 1], ["Legitimate", "Phishing"])
plt.yticks([0, 1], ["Legitimate", "Phishing"])
for (i, j), value in np.ndenumerate(cm):
    plt.text(j, i, str(value), ha="center", va="center")
plt.tight_layout()
plt.show()

## 5. Precision, recall and F1; domain error trade-off
Phishing is the positive class. In this context, a false negative means a phishing URL is missed and treated as legitimate. A false positive means a legitimate URL is incorrectly flagged. For a security-warning tool, false negatives can expose clinic staff to malicious sites, so they may be more harmful; however, excessive false positives can create alert fatigue and reduce trust. The appropriate threshold depends on how the system is deployed.

In [ ]:
print("Precision:", round(precision_score(y_test, best_predictions, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, best_predictions, zero_division=0), 4))
print("F1:", round(f1_score(y_test, best_predictions, zero_division=0), 4))
print("False negative count:", fn, "| False positive count:", fp)

## 6. Five-fold cross-validation
Run stratified 5-fold cross-validation on the training data and compare the mean and standard deviation of F1 scores. Variation between folds indicates how sensitive the result is to the particular split.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_rows = []
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1)
    cv_rows.append({
        "Model": name,
        "Fold F1 scores": np.round(scores, 4).tolist(),
        "Mean CV F1": scores.mean(),
        "Std CV F1": scores.std()
    })
cv_table = pd.DataFrame(cv_rows).set_index("Model")
display(cv_table)

## 7. Three-sentence model recommendation
Use the evaluation table and cross-validation table above to choose the model. Complete this evidence-based recommendation after running the notebook; do not choose a model based only on accuracy.

In [ ]:
recommended = evaluation_table.index[0]
print(f"1. Based on the held-out F1 table, {recommended} is the leading candidate in this run.")
print("2. Its limitations include possible dataset bias, URL-pattern changes over time, and errors on domains not represented in the sample.")
print("3. I would use it only as a human-reviewed warning aid after domain-separated external testing, threshold tuning, and ongoing monitoring; it should not automatically block or accuse users.")

## Completion checklist
- [ ] Confusion matrix produced and all four cells explained.
- [ ] Precision, recall and F1 computed.
- [ ] False-positive versus false-negative cost argued for cybersecurity.
- [ ] Stratified 5-fold cross-validation run and compared with hold-out results.
- [ ] Preprocessing and estimator are combined in a pipeline.
- [ ] Evaluation table compares at least three models on the same metrics.
- [ ] Three-sentence recommendation includes weaknesses.